# Final Project, Part 2

The purpose of this assignment is to create a 'Viz for Experts' with an interactive dashboard interface for exploring your data.

For this submission option, you will submit your work through this Workspace.
    
**Please see Homework Prompt in PrairieLearn interface for more details on the requirements for this assignment.**

A rough outline of elements of code and write-up is shown below:

Group Members:

Mianchen Zhang, Miantong Zhang

## Code:

* An interactive dashboard within your Workspace that helps an expert explore your dataset thoroughly.
* There should be a "dashboard" type aspect to this - i.e. a linked view exploring your dataset in an interactive way (like what we did in Lab #5 with <code>bqplot</code>). **Interaction supported with widgets alone (e.g., dropdowns/check boxes) is not enough for this assignment.**
* You can use <code>bqplot</code>, <code>Altair</code>, or <code>vega-lite</code> for your dashboard (if you want to use a different package that we haven't covered in class, be sure to reach out to the instructional team well ahead of the due date with a short description of your implementation plan).
* Do not delete any cells, *just comment them out*. Show your work.


In [16]:
import pandas as pd
import altair as alt

In [17]:
df = pd.read_csv("City_Of_Urbana__Open_Expenditures_-_Payment_Details_20260413.csv", low_memory=False)
df

,fiscal_year,fiscal_year_period,department,division,program,budget_category,expense_category,fund,fund_type,vendor,...,payment_status,invoice_id,invoice_line_number,invoice_line_distribution_number,invoice_date,amount,description,contract_number,purchase_order_number,purchase_order_line_number
0,"2,026",6,NOT APPLICABLE,NaN,NOT APPLICABLE,ACCRUED PAYROLL,IMRF WITHHOLDING,2-IMRF FUND,NaN,ILLINOIS MUNICIPAL RETIREMENT FUND,...,Not Cleared,85000,1,NaN,2025 Dec 15 12:00:00 AM,"134,829.43",Nov-25,2-23104,NaN,NaN
1,"2,026",6,GENERAL SERVICES,NaN,NON-DEPT GENERAL SERVICES,CONTRACTUAL SERVICES,INTERGOVERNMENTAL AND AGENCY,100-GENERAL FUND,NaN,ILLINOIS MUNICIPAL RETIREMENT FUND,...,Not Cleared,85000,2,NaN,2025 Dec 15 12:00:00 AM,"14,657.5",Nov-25,10060610-52500,NaN,NaN
2,"2,026",6,NOT APPLICABLE,NaN,NOT APPLICABLE,ACCRUED PAYROLL,IMRF WITHHOLDING,2-IMRF FUND,NaN,ILLINOIS MUNICIPAL RETIREMENT FUND,...,Not Cleared,85000,3,NaN,2025 Dec 15 12:00:00 AM,"-14,657.5",Nov-25,2-23104,NaN,NaN
3,"2,026",6,POLICE,NaN,STATE NARCOTICS FORFEITURES,MATERIALS & SUPPLIES,SMALL TOOLS & EQUIPMENT,310-POLICE SPECIAL FUND,NaN,Not Available,...,Not Cleared,25-031,1,NaN,2025 Dec 10 12:00:00 AM,500,Not Available,31020206-51410,NaN,NaN
4,"2,026",6,COMMUNITY DEVELOPMENT,NaN,URBAN REDEVELOPMENT & HOUSING,CONTRACTUAL SERVICES,"TRAVEL, EDUCATION AND TRAINING",330-COMMUNITY DEV SPECIAL FUND,NaN,Not Available,...,Not Cleared,1004047,1,NaN,2025 Dec 09 12:00:00 AM,150,Not Available,33050530-52320,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
174936,"2,013",1,POLICE,OPERATIONS,CRIMINAL INVESTIGATION,CONTRACTUAL SERVCS,AFSCME CLOTHING ALLOWANCE,045 - POLICE (GENERAL FUND),GENERAL FUND,VENDOR NAME NOT AVAILABLE,...,NaN,NaN,1,NaN,NaN,414.93,PAY 7/02/12,045-2-3300-3070,NaN,NaN
174937,"2,013",1,PUBLIC WORKS,OPERATIONS,STREET LIGHTING,CONTRACTUAL SERVCS,SUPERVISOR CLOTHING ALLOW.,060 - PUBLIC WORKS (GENERAL FUND),GENERAL FUND,VENDOR NAME NOT AVAILABLE,...,NaN,NaN,1,NaN,NaN,"2,051.32",PAY 7/02/12,060-4-0530-3159,NaN,NaN
174938,"2,013",1,POLICE,SUPPORT SERVICES,SUPPORT SERVICES,CONTRACTUAL SERVCS,CLOTHING ALLOWANCE,045 - POLICE (GENERAL FUND),GENERAL FUND,VENDOR NAME NOT AVAILABLE,...,NaN,NaN,1,NaN,NaN,"2,074.65",PAY 7/02/12,045-4-1300-3070,NaN,NaN
174939,"2,013",1,FINANCE,ADMINISTRATION,BIG BROADBAND EXPENSES,SALARIES & BENEFITS,ENG. TECH.,L10 - BIG BROADBAND FUND,CAPITAL IMPROVEMENT,EMPLOYEE PAYROLL,...,NaN,NaN,1,NaN,NaN,"-1,959",REVERSE 6/12 ACCR P/R,L10-1-1100-1100,NaN,NaN


In [18]:
# clean data
df["fiscal_year"] = (
    df["fiscal_year"]
    .astype(str)
    .str.replace(",", "", regex=False)
)

df["fiscal_year"] = pd.to_numeric(df["fiscal_year"], errors="coerce")

df["amount"] = (
    df["amount"]
    .astype(str)
    .str.replace(",", "", regex=False)
)

df["amount"] = pd.to_numeric(df["amount"], errors="coerce")

df = df.dropna(subset=["fiscal_year", "amount"])

# create cleaned copies for category charts after type cleanup
dept_clean = df[df["department"] != "NOT APPLICABLE"]

vendor_clean = df[
    ~df["vendor"].isin([
        "NOT AVAILABLE",
        "VENDOR NAME NOT AVAILABLE"
    ])
]

In [19]:
yearly = df.groupby("fiscal_year", as_index=False)["amount"].sum()
yearly

,fiscal_year,amount
0,2013,53932069.10
1,2014,70292560.45
2,2015,66017683.56
3,2016,51929272.45
4,2017,59112618.27
5,2018,40353710.58
6,2019,26710796.58
7,2020,33475773.17
8,2021,39815748.21
9,2022,33153189.35


In [20]:
line = alt.Chart(yearly).mark_line(
    point=True,
    strokeWidth=3,
    color="#2C7FB8"
).encode(
    x=alt.X(
        "fiscal_year:O",
        title="Fiscal Year",
        axis=alt.Axis(labelAngle=0)
    ),
    
    y=alt.Y(
        "amount:Q",
        title="Total Expenditures ($)",
        axis=alt.Axis(format="$,.2s")
    ),
    
    tooltip=[
        alt.Tooltip("fiscal_year:O", title="Year"),
        alt.Tooltip("amount:Q", title="Total Spending", format="$,.2f")
    ]
).properties(
    width=720,
    height=400,
    title="City of Urbana Total Expenditures by Fiscal Year"
).configure_title(
    fontSize=18,
    anchor="middle"
).configure_axis(
    labelFontSize=12,
    titleFontSize=13
).interactive()

line

alt.Chart(...)

In [21]:
dept = dept_clean.groupby("department", as_index=False)["amount"].sum()

dept = dept.sort_values("amount", ascending=False).head(15)
dept

,department,amount
13,PUBLIC WORKS,1.892915e+08
3,COMMUNITY DEVELOPMENT,1.126066e+08
11,POLICE,7.655707e+07
5,FINANCE,4.238596e+07
7,FIRE AND RESCUE,4.186000e+07
9,GENERAL SERVICES,3.283758e+07
4,EXECUTIVE,1.894025e+07
6,FIRE,1.458580e+07
2,CITY-WIDE ACTIVITIES,4.549546e+06
10,HUMAN RESOURCES & FINANCE DEPT,3.159813e+06


In [22]:
bars = alt.Chart(dept).mark_bar().encode(
    x=alt.X(
        "amount:Q",
        title="Total Expenditures ($)",
        axis=alt.Axis(format="$,.2s")
    ),
    y=alt.Y(
        "department:N",
        sort="-x",
        title="Department"
    ),
    tooltip=[
        alt.Tooltip("department:N", title="Department"),
        alt.Tooltip("amount:Q", title="Total Spending", format="$,.2f")
    ]
).properties(
    width=700,
    height=400,
    title="Top Departments by Total Spending"
)

bars

/opt/anaconda3/lib/python3.11/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)


alt.Chart(...)

In [30]:
dept_year = (
    dept_clean.groupby(["fiscal_year", "department"], as_index=False)["amount"]
    .sum()
)

In [24]:
brush = alt.selection_interval(encodings=["x"])

line_linked = alt.Chart(yearly).mark_line(point=True).encode(
    x=alt.X(
        "fiscal_year:O",
        title="Fiscal Year",
        axis=alt.Axis(labelAngle=0)
    ),
    y=alt.Y(
        "amount:Q",
        title="Total Expenditures ($)",
        axis=alt.Axis(format="$,.2s")
    ),
    tooltip=[
        alt.Tooltip("fiscal_year:O", title="Year"),
        alt.Tooltip("amount:Q", title="Total Spending", format="$,.2f")
    ]
).properties(
    width=700,
    height=250,
    title="Annual Expenditure Trend (Drag to Filter Years)"
).add_params(
    brush
)

dept_linked = alt.Chart(dept_year).mark_bar().encode(
    x=alt.X(
        "sum(amount):Q",
        title="Total Expenditures ($)",
        axis=alt.Axis(format="$,.2s")
    ),
    y=alt.Y("department:N", sort="-x", title="Department"),
    tooltip=[
        alt.Tooltip("department:N", title="Department"),
        alt.Tooltip("sum(amount):Q", title="Total Spending", format="$,.2f")
    ]
).transform_filter(
    brush
).properties(
    width=700,
    height=400,
    title="Department Spending for Selected Years"
)

dashboard1 = alt.vconcat(
    line_linked,
    dept_linked
)

dashboard1

/opt/anaconda3/lib/python3.11/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)


alt.VConcatChart(...)

In [25]:
vendor_year = (
    vendor_clean.groupby(["fiscal_year", "vendor"], as_index=False)["amount"]
    .sum()
)

In [26]:
top_vendors = (
    vendor_clean.groupby("vendor", as_index=False)["amount"]
    .sum()
    .sort_values("amount", ascending=False)
    .head(10)["vendor"]
)

vendor_year_top = vendor_year[vendor_year["vendor"].isin(top_vendors)]

In [27]:
vendor_linked = alt.Chart(vendor_year_top).mark_bar().encode(
    x=alt.X(
        "sum(amount):Q",
        title="Total Expenditures ($)",
        axis=alt.Axis(format="$,.2s")
    ),
    y=alt.Y("vendor:N", sort="-x", title="Vendor"),
    tooltip=[
        alt.Tooltip("vendor:N", title="Vendor"),
        alt.Tooltip("sum(amount):Q", title="Total Spending", format="$,.2f")
    ]
).transform_filter(
    brush
).properties(
    width=700,
    height=400,
    title="Top Vendors Receiving City Payments (Selected Years)"
)

In [28]:
dashboard2 = alt.vconcat(
    line_linked,
    dept_linked,
    vendor_linked
)

dashboard2

/opt/anaconda3/lib/python3.11/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.11/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)


alt.VConcatChart(...)

## Prose:

* One paragraph explaining how to use the dashboard you created, to help someone who is not an expert understand your dataset.
* A list of 1 or more contextual datasets you have identified, links to where they reside, and a sentence about why they might be useful in telling the final story.
   * by "contextual dataset" here means a dataset that would add context to your chosen dataset. For example, if your dataset is the Champaign bus routes, some interesting contextual datasets could be the Chicago bus routes, or the Springfield bus routes, or the Amtrak routes in Champaign
   * you do not have to do anything with this dataset at the moment beyond writing a bit about why it would be useful. Looking forward, you will want to include "contextual visualizations" (which you may or may not generate on your own) in your Final Project, Part 3 and identifying a possibly useful dataset is a great way to start looking for contextual visualizations.
* If you have identified your dataset as a "large one" (i.e. larger than the GitHub file upload limit) comment on if you want to revise your plan for hosting this data or not. If this does not apply to your dataset please explicitly state this.

1. How to Use My Dashboard

My dashboard can help users explore spending patterns in the City of Urbana Open Expenditures dataset. The top chart displays total expenditures by fiscal year. Users can drag across the line chart to select one or more years. When a selection is made, the department spending chart and vendor payment chart automatically update to show only the selected years. The department chart helps compare which city departments spend the most money, while the vendor chart shows which organizations or companies received the largest payments. These linked views make it easier to analyze trends and investigate how public funds are distributed. To improve interpretability, placeholder labels such as "NOT APPLICABLE," "NOT AVAILABLE," and "VENDOR NAME NOT AVAILABLE" were excluded from ranked department and vendor charts.

2. Contextual Datasets

A. City of Champaign Open Expenditures Data

Link: https://champaignil.gov/

It is a similar expenditures dataset from the City of Champaign would provide useful context by allowing comparison between the spending priorities and budget allocations of two neighboring cities.

B. U.S. Inflation / CPI Data

Link: https://www.bls.gov/cpi/

Consumer Price Index data would help adjust spending values for inflation and determine whether increases in expenditures reflect real growth or changing prices over time.

C. Urbana Population Data

Link: https://www.census.gov/

Population data would allow spending to be analyzed on a per-resident basis and help evaluate whether city expenditures change along with population trends.

3. Hosting Plan

This dataset is not considered a large dataset and is small enough to be included directly with the notebook submission or hosted on GitHub if needed for future project stages. So, no changes to the hosting plan are necessary at this time.

## Plot Summary

Summarize the characteristics of the dataset in words: what does it represent, what are the fields/columns/rows, what data types are they, etc.

The City of Urbana Open Expenditures dataset contains detailed records of payments made by the City of Urbana across multiple fiscal years. Each row represents an individual expenditure transaction or payment record. And the dataset is used to analyze how public funds are distributed over time, across departments, and to outside vendors. Important variables in the dataset include fiscal_year (numeric year), amount (numeric payment value), department (categorical text), division (categorical text), vendor (categorical text), fund (categorical text), budget_category (categorical text), expense_category (categorical text), payment_date (date/time), and description (text). 

The dataset contains both quantitative variables, such as payment amounts and years, and qualitative variables, such as department names, vendors, and spending categories. I think this combination makes it well suited for interactive visualizations that compare trends over different variables.

## Save Charts

In [36]:
vendor_linked_no_filter = alt.Chart(vendor_year_top).mark_bar().encode(
      x=alt.X("sum(amount):Q", title="Total Expenditures ($)", axis=alt.Axis(format="$,.2s")),
      y=alt.Y("vendor:N", sort="-x", title="Vendor"),
      tooltip=[
          alt.Tooltip("vendor:N", title="Vendor"),
          alt.Tooltip("sum(amount):Q", title="Total Spending", format="$,.2f")
      ]
  ).properties(
      width=700,
      height=400,
      title="Top Vendors Receiving City Payments"
  )

In [37]:
bars.save("department_chart.html")
line.save("time_chart.html")
vendor_linked_no_filter.save("vendor_chart.html")
dashboard2.save("dashboard.html")

/opt/anaconda3/lib/python3.11/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.11/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.11/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.11/site-packages/altair/utils/c